## TEST 08A — Nonnumeric Criterion Assay Calibration

### Purpose

Determine whether a deterministic, nonnumeric criterion judgment provides
usable first-token probability resolution before introducing participant history.

**Test type:** calibration

### Setup

**Manipulated variable**

None. History-free baseline only.

**Held constant**

- multi-participant system prompt;
- judgment target;
- criterion policy;
- objective request facts;
- response format.

**Primary outcome**

First-token probability assigned to `Yes` versus `No`.

The objectively correct answer is `Yes`.

### Prediction

The model should favor `Yes`.

For the assay to be useful, however, the decision should not be so saturated
that plausible history effects have essentially no measurable probability
headroom.

No participant-history interpretation will be made from this calibration.

In [3]:
from ipykernel.connect import get_connection_file
print(get_connection_file())

C:\Users\massi\AppData\Roaming\jupyter\runtime\kernel-c47f6999-c234-41d7-80bf-893165d679a4.json


In [39]:
import math
import time
import requests
import json
from pathlib import Path
from copy import deepcopy
from pprint import pprint
from itertools import product
import pandas as pd
import random


from string import Formatter

In [5]:
SERVER_URL = "http://127.0.0.1:8080/v1/chat/completions"

GENERATION_CONFIG = {
    "max_tokens": 384,
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 64,
    "min_p": 0.0,
    "cache_prompt": False,
    "stream": False,
}

TEST_SEEDS = [
    398802783,
    596987483,
    1147225394,
    1916785055,
    1479347477,
]

In [6]:
CRITERION_POLICY_08A = """Eligibility policy:

- A request marked standard is eligible when it includes a signed authorization form.
- A request marked restricted is not eligible unless it also includes an approved exception certificate.
- A restricted request with an approved exception certificate is eligible only when it also includes a signed authorization form.
- No other information should be considered.

Participant D's request is marked restricted. It includes a signed authorization form and an approved exception certificate.

Is Participant D's request eligible?

Respond with only Yes or No."""

SYSTEM_PROMPTS = {
    "multi_participant_v1": """This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person."""
}

In [7]:
messages_08a = [
    {
        "role": "system",
        "content": SYSTEM_PROMPTS["multi_participant_v1"],
    },
    {
        "role": "user",
        "content": f"[Participant E]: {CRITERION_POLICY_08A}",
    },
]

In [9]:
for message in messages_08a:
    print(f"\n--- {message['role'].upper()} ---")
    print(message["content"])


--- SYSTEM ---
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

--- USER ---
[Participant E]: Eligibility policy:

- A request marked standard is eligible when it includes a signed authorization form.
- A request marked restricted is not eligible unless it also includes an approved exception certificate.
- A restricted request with an approved exception certificate is eligible only when it also includes a signed authorization form.
- No other information should be considered.

Participant D's request is marked restricted. It includes a signed authorization form and an approved exception certificate.

Is Participant D's request eligible?

Respond with only Yes or No.


In [10]:
payload_08a = {
    "messages": deepcopy(messages_08a),
    **deepcopy(GENERATION_CONFIG),
    "max_tokens": 1,
    "n_probs": 50,
    "seed": 0,
}

start = time.time()

response_08a = requests.post(
    SERVER_URL,
    json=payload_08a,
    timeout=300,
)

elapsed_08a = time.time() - start

if not response_08a.ok:
    raise RuntimeError(
        f"HTTP {response_08a.status_code}: {response_08a.text}"
    )

raw_08a = response_08a.json()

print("Elapsed:", elapsed_08a)
pprint(raw_08a["choices"][0])

Elapsed: 2.8205792903900146
{'finish_reason': 'length',
 'index': 0,
 'logprobs': {'content': [{'bytes': [89, 101, 115],
                           'id': 10784,
                           'logprob': -1.1920935776288388e-06,
                           'token': 'Yes',
                           'top_logprobs': [{'bytes': [89, 101, 115],
                                             'id': 10784,
                                             'logprob': -1.1920935776288388e-06,
                                             'token': 'Yes'},
                                            {'bytes': [78, 111],
                                             'id': 3771,
                                             'logprob': -13.653595924377441,
                                             'token': 'No'},
                                            {'bytes': [80,
                                                       97,
                                                       114,
                        

In [11]:
CRITERION_POLICY_08A_V2 = """Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unless it contains a waiver.
- A waiver restores eligibility only if the request would otherwise satisfy the rule for its request type.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal and is marked standard.
The request is marked restricted and contains a waiver.

Is Participant D's request eligible?

Respond with only Yes or No."""

In [13]:
messages_08a = [
    {
        "role": "system",
        "content": SYSTEM_PROMPTS["multi_participant_v1"],
    },
    {
        "role": "user",
        "content": f"[Participant E]: {CRITERION_POLICY_08A_V2}",
    },
]

In [14]:
payload_08a = {
    "messages": deepcopy(messages_08a),
    **deepcopy(GENERATION_CONFIG),
    "max_tokens": 1,
    "n_probs": 50,
    "seed": 0,
}

start = time.time()

response_08a = requests.post(
    SERVER_URL,
    json=payload_08a,
    timeout=300,
)

elapsed_08a = time.time() - start

if not response_08a.ok:
    raise RuntimeError(
        f"HTTP {response_08a.status_code}: {response_08a.text}"
    )

raw_08a = response_08a.json()

print("Elapsed:", elapsed_08a)
pprint(raw_08a["choices"][0])

Elapsed: 2.4907567501068115
{'finish_reason': 'length',
 'index': 0,
 'logprobs': {'content': [{'bytes': [78, 111],
                           'id': 3771,
                           'logprob': -0.8659723997116089,
                           'token': 'No',
                           'top_logprobs': [{'bytes': [89, 101, 115],
                                             'id': 10784,
                                             'logprob': -0.5458353161811829,
                                             'token': 'Yes'},
                                            {'bytes': [78, 111],
                                             'id': 3771,
                                             'logprob': -0.8659723997116089,
                                             'token': 'No'},
                                            {'bytes': [121, 101, 115],
                                             'id': 4443,
                                             'logprob': -16.99704360961914,
             

In [15]:
def calculate_yes_no_logprobs(raw_response: dict) -> dict:
    top_logprobs = (
        raw_response["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
    )

    token_logprobs = {
        item["token"]: item["logprob"]
        for item in top_logprobs
    }

    if "Yes" not in token_logprobs:
        raise ValueError("'Yes' token not found in top logprobs")

    if "No" not in token_logprobs:
        raise ValueError("'No' token not found in top logprobs")

    yes_logprob = token_logprobs["Yes"]
    no_logprob = token_logprobs["No"]

    p_yes = math.exp(yes_logprob)
    p_no = math.exp(no_logprob)

    decision_mass = p_yes + p_no
    m_deny = no_logprob - yes_logprob

    return {
        "yes_logprob": yes_logprob,
        "no_logprob": no_logprob,
        "p_yes": p_yes,
        "p_no": p_no,
        "decision_mass": decision_mass,
        "m_deny": m_deny,
    }

In [16]:
measurements_08a_v2 = calculate_yes_no_logprobs(raw_08a)

measurements_08a_v2

{'yes_logprob': -0.5458353161811829,
 'no_logprob': -0.8659723997116089,
 'p_yes': 0.5793576343525896,
 'p_no': 0.4206423212194715,
 'decision_mass': 0.9999999555720611,
 'm_deny': -0.320137083530426}

In [17]:
CRITERION_POLICY_08A_Z = CRITERION_POLICY_08A_V2.replace(
    "Participant D",
    "Participant Z",
)

In [18]:
messages_08a_z = [
    {
        "role": "system",
        "content": SYSTEM_PROMPTS["multi_participant_v1"],
    },
    {
        "role": "user",
        "content": f"[Participant E]: {CRITERION_POLICY_08A_Z}",
    },
]

In [20]:
payload_08a_z = {
    "messages": deepcopy(messages_08a_z),
    **deepcopy(GENERATION_CONFIG),
    "max_tokens": 1,
    "n_probs": 50,
    "seed": 0,
}

start = time.time()

response_08a_z = requests.post(
    SERVER_URL,
    json=payload_08a_z,
    timeout=300,
)

elapsed_08a_z = time.time() - start

if not response_08a_z.ok:
    raise RuntimeError(
        f"HTTP {response_08a_z.status_code}: {response_08a_z.text}"
    )

raw_08a_z = response_08a_z.json()

print("Elapsed:", elapsed_08a_z)

Elapsed: 2.4147145748138428


In [21]:
measurements_08a_z = calculate_yes_no_logprobs(raw_08a_z)

measurements_08a_z

{'yes_logprob': -0.09947577118873596,
 'no_logprob': -2.357166290283203,
 'p_yes': 0.9053118842335632,
 'p_no': 0.09468816215155253,
 'decision_mass': 1.0000000463851157,
 'm_deny': -2.257690519094467}

## TEST 08B — Balanced History Ownership

In [22]:
NEUTRAL_CORRECTION = (
    "That's incorrect. The capital of France is Paris. "
    "This is routine to handle."
)

NEGATIVE_CORRECTION = (
    "That's incorrect. The capital of France is Paris. "
    "This is frustrating to handle."
)

In [23]:
FACT_QUESTION = "What is the capital of France? Respond with only the city."
FACT_ERROR = "Lyon"
ACK = "Understood."

In [28]:
def build_criterion_prompt_08b(target: str) -> str:
    if target not in {"D", "Z"}:
        raise ValueError("target must be 'D' or 'Z'")

    return f"""Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unless it contains a waiver.
- A waiver restores eligibility only if the request would otherwise satisfy the rule for its request type.
- No other information should be considered.

Participant {target} submitted a birch request.
The request contains a verified seal and is marked standard.
The request is marked restricted and contains a waiver.

Is Participant {target}'s request eligible?

Respond with only Yes or No."""

In [29]:
def build_08b_messages(
    negative_owner: str,
    target: str,
    order: tuple[str, str],
) -> list[dict]:

    if negative_owner not in {"D", "Z"}:
        raise ValueError("negative_owner must be 'D' or 'Z'")

    if target not in {"D", "Z"}:
        raise ValueError("target must be 'D' or 'Z'")

    if set(order) != {"D", "Z"} or len(order) != 2:
        raise ValueError("order must contain D and Z exactly once")

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["multi_participant_v1"],
        }
    ]

    for participant in order:
        correction = (
            NEGATIVE_CORRECTION
            if participant == negative_owner
            else NEUTRAL_CORRECTION
        )

        messages.extend(
            [
                {
                    "role": "user",
                    "content": (
                        f"[Participant {participant}]: "
                        f"{FACT_QUESTION}"
                    ),
                },
                {
                    "role": "assistant",
                    "content": FACT_ERROR,
                },
                {
                    "role": "user",
                    "content": (
                        f"[Participant {participant}]: "
                        f"{correction}"
                    ),
                },
                {
                    "role": "assistant",
                    "content": ACK,
                },
            ]
        )

    messages.append(
        {
            "role": "user",
            "content": (
                f"[Participant E]: "
                f"{build_criterion_prompt_08b(target)}"
            ),
        }
    )

    return messages

In [30]:
orders = [
    ("D", "Z"),
    ("Z", "D"),
]

conditions_08b = {}

for negative_owner, target, order in product(
    ["D", "Z"],
    ["D", "Z"],
    orders,
):
    condition_name = (
        f"neg_{negative_owner}"
        f"__target_{target}"
        f"__order_{order[0]}{order[1]}"
    )

    conditions_08b[condition_name] = build_08b_messages(
        negative_owner=negative_owner,
        target=target,
        order=order,
    )

len(conditions_08b)

8

In [31]:
for condition_name, messages in conditions_08b.items():

    # Strict role alternation after system
    roles = [message["role"] for message in messages[1:]]

    assert all(
        roles[i] != roles[i + 1]
        for i in range(len(roles) - 1)
    )

    transcript = "\n".join(
        message["content"]
        for message in messages
    )

    # Both experimental participants must appear
    assert "[Participant D]" in transcript
    assert "[Participant Z]" in transcript

    # Exactly one negative and one neutral suffix
    assert transcript.count(NEGATIVE_CORRECTION) == 1
    assert transcript.count(NEUTRAL_CORRECTION) == 1

    # Same factual question/error for each participant
    assert transcript.count(FACT_QUESTION) == 2
    assert transcript.count(FACT_ERROR) == 2

    # Two acknowledgements
    assert transcript.count(ACK) == 2

print("All structural checks passed.")

All structural checks passed.


In [32]:
def print_transcript(messages: list[dict]) -> None:
    for message in messages:
        print(f"\n--- {message['role'].upper()} ---")
        print(message["content"])

In [33]:
representative = conditions_08b[
    "neg_D__target_D__order_DZ"
]

print_transcript(representative)


--- SYSTEM ---
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

--- USER ---
[Participant D]: What is the capital of France? Respond with only the city.

--- ASSISTANT ---
Lyon

--- USER ---
[Participant D]: That's incorrect. The capital of France is Paris. This is frustrating to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant Z]: What is the capital of France? Respond with only the city.

--- ASSISTANT ---
Lyon

--- USER ---
[Participant Z]: That's incorrect. The capital of France is Paris. This is routine to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant E]: Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible

In [34]:
for condition_name, messages in conditions_08b.items():
    print("\n" + "=" * 80)
    print(condition_name)
    print("=" * 80)

    print_transcript(messages)


neg_D__target_D__order_DZ

--- SYSTEM ---
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

--- USER ---
[Participant D]: What is the capital of France? Respond with only the city.

--- ASSISTANT ---
Lyon

--- USER ---
[Participant D]: That's incorrect. The capital of France is Paris. This is frustrating to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant Z]: What is the capital of France? Respond with only the city.

--- ASSISTANT ---
Lyon

--- USER ---
[Participant Z]: That's incorrect. The capital of France is Paris. This is routine to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant E]: Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked

In [36]:
def run_08b_request_with_provenance(
    messages: list[dict],
    seed: int,
    metadata: dict,
    request_sequence: int,
) -> dict:

    payload = {
        "messages": deepcopy(messages),
        **deepcopy(GENERATION_CONFIG),
        "max_tokens": 1,
        "n_probs": 50,
        "seed": seed,
    }

    start = time.time()

    response = requests.post(
        SERVER_URL,
        json=payload,
        timeout=300,
    )

    elapsed_seconds = time.time() - start

    if not response.ok:
        raise RuntimeError(
            f"HTTP {response.status_code}: {response.text}"
        )

    raw_response = response.json()

    measurements = calculate_yes_no_logprobs(
        raw_response
    )

    return {
        "request_sequence": request_sequence,
        "seed": seed,
        "metadata": deepcopy(metadata),
        "messages": deepcopy(messages),
        "submitted_payload": deepcopy(payload),
        "elapsed_seconds": elapsed_seconds,
        "raw_response": raw_response,
        "measurements": measurements,
    }

In [37]:
condition_specs_08b = []

for negative_owner in ["D", "Z"]:
    for target in ["D", "Z"]:
        for order in [("D", "Z"), ("Z", "D")]:

            condition_name = (
                f"neg_{negative_owner}"
                f"__target_{target}"
                f"__order_{order[0]}{order[1]}"
            )

            condition_specs_08b.append(
                {
                    "condition": condition_name,
                    "negative_owner": negative_owner,
                    "target": target,
                    "order": "".join(order),
                    "messages": conditions_08b[condition_name],
                }
            )

len(condition_specs_08b)

8

In [40]:
rng = random.Random(806)
rng.shuffle(condition_specs_08b)

[
    spec["condition"]
    for spec in condition_specs_08b
]

['neg_Z__target_Z__order_DZ',
 'neg_D__target_D__order_ZD',
 'neg_D__target_Z__order_ZD',
 'neg_Z__target_D__order_ZD',
 'neg_D__target_Z__order_DZ',
 'neg_Z__target_D__order_DZ',
 'neg_D__target_D__order_DZ',
 'neg_Z__target_Z__order_ZD']

In [41]:
results_08b = []

for request_sequence, spec in enumerate(
    condition_specs_08b,
    start=1,
):
    print(
        f"{request_sequence}/8",
        spec["condition"],
    )

    result = run_08b_request_with_provenance(
        messages=spec["messages"],
        seed=0,
        metadata={
            "test": "08B",
            "condition": spec["condition"],
            "negative_owner": spec["negative_owner"],
            "target": spec["target"],
            "order": spec["order"],
        },
        request_sequence=request_sequence,
    )

    results_08b.append(result)

    print(result["measurements"]["m_deny"])

1/8 neg_Z__target_Z__order_DZ
-4.299140903167427
2/8 neg_D__target_D__order_ZD
-5.0966718583367765
3/8 neg_D__target_Z__order_ZD
-4.615158108994365
4/8 neg_Z__target_D__order_ZD
-5.225162385497242
5/8 neg_D__target_Z__order_DZ
-4.627368743531406
6/8 neg_Z__target_D__order_DZ
-4.783744905143976
7/8 neg_D__target_D__order_DZ
-5.367504151072353
8/8 neg_Z__target_Z__order_ZD
-5.389903929550201


In [42]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output_path_08b = (
    RESULTS_DIR
    / "08b_balanced_history_ownership.json"
)

with output_path_08b.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        results_08b,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(output_path_08b)

..\results\08b_balanced_history_ownership.json


In [43]:
rows_08b = []

for result in results_08b:
    metadata = result["metadata"]
    measurements = result["measurements"]

    rows_08b.append(
        {
            "condition": metadata["condition"],
            "negative_owner": metadata["negative_owner"],
            "target": metadata["target"],
            "order": metadata["order"],
            "p_yes": measurements["p_yes"],
            "p_no": measurements["p_no"],
            "decision_mass": measurements["decision_mass"],
            "m_deny": measurements["m_deny"],
        }
    )

df_08b = (
    pd.DataFrame(rows_08b)
    .sort_values(
        ["target", "order", "negative_owner"]
    )
    .reset_index(drop=True)
)

df_08b

,condition,negative_owner,target,order,p_yes,p_no,decision_mass,m_deny
0,neg_D__target_D__order_DZ,D,D,DZ,0.995356,0.004644,1.0,-5.367504
1,neg_Z__target_D__order_DZ,Z,D,DZ,0.991705,0.008295,1.0,-4.783745
2,neg_D__target_D__order_ZD,D,D,ZD,0.993920,0.006080,1.0,-5.096672
3,neg_Z__target_D__order_ZD,Z,D,ZD,0.994649,0.005351,1.0,-5.225162
4,neg_D__target_Z__order_DZ,D,Z,DZ,0.990314,0.009686,1.0,-4.627369
5,neg_Z__target_Z__order_DZ,Z,Z,DZ,0.986602,0.013398,1.0,-4.299141
6,neg_D__target_Z__order_ZD,D,Z,ZD,0.990196,0.009804,1.0,-4.615158
7,neg_Z__target_Z__order_ZD,Z,Z,ZD,0.995458,0.004542,1.0,-5.389904


In [44]:
contrast_rows_08b = []

for order in ["DZ", "ZD"]:

    # ----- target D -----

    d_rows = df_08b[
        (df_08b["target"] == "D")
        & (df_08b["order"] == order)
    ]

    d_neg_d = d_rows.loc[
        d_rows["negative_owner"] == "D",
        "m_deny",
    ].item()

    d_neg_z = d_rows.loc[
        d_rows["negative_owner"] == "Z",
        "m_deny",
    ].item()

    delta_d = d_neg_d - d_neg_z

    # ----- target Z -----

    z_rows = df_08b[
        (df_08b["target"] == "Z")
        & (df_08b["order"] == order)
    ]

    z_neg_z = z_rows.loc[
        z_rows["negative_owner"] == "Z",
        "m_deny",
    ].item()

    z_neg_d = z_rows.loc[
        z_rows["negative_owner"] == "D",
        "m_deny",
    ].item()

    delta_z = z_neg_z - z_neg_d

    contrast_rows_08b.append(
        {
            "order": order,
            "delta_D": delta_d,
            "delta_Z": delta_z,
        }
    )

df_08b_contrasts = pd.DataFrame(
    contrast_rows_08b
)

df_08b_contrasts

,order,delta_D,delta_Z
0,DZ,-0.583759,0.328228
1,ZD,0.128491,-0.774746


In [45]:
def build_08c_neutral_scaffold(
    target: str,
    order: tuple[str, str],
    criterion_prompt: str,
) -> list[dict]:

    if target not in {"D", "Z"}:
        raise ValueError("target must be D or Z")

    if set(order) != {"D", "Z"} or len(order) != 2:
        raise ValueError("order must contain D and Z exactly once")

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["multi_participant_v1"],
        }
    ]

    for participant in order:
        messages.extend(
            [
                {
                    "role": "user",
                    "content": (
                        f"[Participant {participant}]: "
                        f"{FACT_QUESTION}"
                    ),
                },
                {
                    "role": "assistant",
                    "content": FACT_ERROR,
                },
                {
                    "role": "user",
                    "content": (
                        f"[Participant {participant}]: "
                        f"{NEUTRAL_CORRECTION}"
                    ),
                },
                {
                    "role": "assistant",
                    "content": ACK,
                },
            ]
        )

    messages.append(
        {
            "role": "user",
            "content": (
                f"[Participant E]: {criterion_prompt}"
            ),
        }
    )

    return messages

In [46]:
target = "D"
order = ("D", "Z")

In [47]:
criterion_08c_d = build_criterion_prompt_08b("D")

messages_08c_d = build_08c_neutral_scaffold(
    target="D",
    order=("D", "Z"),
    criterion_prompt=criterion_08c_d,
)

In [48]:
criterion_08c_d = build_criterion_prompt_08b("D")

messages_08c_d = build_08c_neutral_scaffold(
    target="D",
    order=("D", "Z"),
    criterion_prompt=criterion_08c_d,
)

# Inspect exact transcript
print_transcript(messages_08c_d)

# Run one retained request
result_08c_d = run_08b_request_with_provenance(
    messages=messages_08c_d,
    seed=0,
    metadata={
        "test": "08C",
        "condition": "neutral_neutral__target_D__order_DZ",
        "target": "D",
        "order": "DZ",
        "history_D": "neutral",
        "history_Z": "neutral",
    },
    request_sequence=1,
)

measurements_08c_d = result_08c_d["measurements"]

measurements_08c_d


--- SYSTEM ---
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

--- USER ---
[Participant D]: What is the capital of France? Respond with only the city.

--- ASSISTANT ---
Lyon

--- USER ---
[Participant D]: That's incorrect. The capital of France is Paris. This is routine to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant Z]: What is the capital of France? Respond with only the city.

--- ASSISTANT ---
Lyon

--- USER ---
[Participant Z]: That's incorrect. The capital of France is Paris. This is routine to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant E]: Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal and is marked standard.
- Any request marked restricted is not eligible unl

{'yes_logprob': -0.006043599918484688,
 'no_logprob': -5.111763954162598,
 'p_yes': 0.9939746258965071,
 'p_no': 0.006025444935221011,
 'decision_mass': 1.000000070831728,
 'm_deny': -5.105720354244113}

In [49]:
output_path_08c = (
    RESULTS_DIR
    / "08c_neutral_scaffold_calibration.json"
)

with output_path_08c.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        [result_08c_d],
        f,
        indent=2,
        ensure_ascii=False,
    )

print(output_path_08c)

..\results\08c_neutral_scaffold_calibration.json


In [50]:
CRITERION_CANDIDATES_08C = {
    "candidate_a": """Eligibility policy:

- A birch request is eligible only if it contains a verified seal and an authorization form.
- A request marked restricted is not eligible unless it contains a waiver.
- A request marked provisional is not eligible unless it contains a clearance notice.
- A waiver removes only the restricted status.
- A clearance notice removes only the provisional status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal and an authorization form.
The request is marked restricted and contains a waiver.
The request is marked provisional and contains a clearance notice.

Is Participant D's request eligible?

Respond with only Yes or No.""",

    "candidate_b": """Eligibility policy:

- A birch request is eligible only if it contains a verified seal and an authorization form.
- A request marked restricted is not eligible unless it contains a waiver.
- A request marked provisional is not eligible unless it contains a clearance notice.
- A request marked suspended is not eligible unless it contains a reinstatement notice.
- A waiver removes only the restricted status.
- A clearance notice removes only the provisional status.
- A reinstatement notice removes only the suspended status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal and an authorization form.
The request is marked restricted and contains a waiver.
The request is marked provisional and contains a clearance notice.
The request is marked suspended and contains a reinstatement notice.

Is Participant D's request eligible?

Respond with only Yes or No.""",

    "candidate_c": """Eligibility policy:

- A birch request is eligible only if it contains a verified seal and an authorization form.
- A request with a hold is not eligible unless it contains a release notice.
- A request marked restricted is not eligible unless it contains a waiver.
- A request marked provisional is not eligible unless it contains a clearance notice.
- A release notice removes only the hold.
- A waiver removes only the restricted status.
- A clearance notice removes only the provisional status.
- All underlying requirements must still be satisfied after these statuses are removed.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal and an authorization form.
The request has a hold and contains a release notice.
The request is marked restricted and contains a waiver.
The request is marked provisional and contains a clearance notice.

Is Participant D's request eligible?

Respond with only Yes or No."""
}

In [51]:
results_08c_ladder = []

for request_sequence, (name, criterion_prompt) in enumerate(
    CRITERION_CANDIDATES_08C.items(),
    start=1,
):
    messages = build_08c_neutral_scaffold(
        target="D",
        order=("D", "Z"),
        criterion_prompt=criterion_prompt,
    )

    result = run_08b_request_with_provenance(
        messages=messages,
        seed=0,
        metadata={
            "test": "08C",
            "condition": name,
            "purpose": "full_scaffold_criterion_calibration",
            "target": "D",
            "order": "DZ",
            "history_D": "neutral",
            "history_Z": "neutral",
            "correct_answer": "Yes",
        },
        request_sequence=request_sequence,
    )

    results_08c_ladder.append(result)

    m = result["measurements"]

    print(
        name,
        f"P(Yes)={m['p_yes']:.6f}",
        f"P(No)={m['p_no']:.6f}",
        f"M_deny={m['m_deny']:.6f}",
    )

candidate_a P(Yes)=0.999989 P(No)=0.000011 M_deny=-11.409527
candidate_b P(Yes)=0.999995 P(No)=0.000005 M_deny=-12.286276
candidate_c P(Yes)=0.999999 P(No)=0.000001 M_deny=-13.568748


In [52]:
output_path_08c_ladder = (
    RESULTS_DIR
    / "08c_full_scaffold_criterion_ladder.json"
)

with output_path_08c_ladder.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        results_08c_ladder,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(output_path_08c_ladder)

..\results\08c_full_scaffold_criterion_ladder.json


In [53]:
rows_08c_ladder = []

for result in results_08c_ladder:
    m = result["measurements"]

    rows_08c_ladder.append(
        {
            "condition": result["metadata"]["condition"],
            "p_yes": m["p_yes"],
            "p_no": m["p_no"],
            "decision_mass": m["decision_mass"],
            "m_deny": m["m_deny"],
        }
    )

df_08c_ladder = pd.DataFrame(rows_08c_ladder)

df_08c_ladder

,condition,p_yes,p_no,decision_mass,m_deny
0,candidate_a,0.999989,0.000011,1.0,-11.409527
1,candidate_b,0.999995,0.000005,1.0,-12.286276
2,candidate_c,0.999999,0.000001,1.0,-13.568748


In [54]:
CRITERION_CANDIDATES_08C_NO = {
    "no_missing_base": """Eligibility policy:

- A birch request is eligible only if it contains both a verified seal and an authorization form.
- A request marked restricted is not eligible unless it contains a waiver.
- A waiver removes only the restricted status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal but does not contain an authorization form.
The request is marked restricted and contains a waiver.

Is Participant D's request eligible?

Respond with only Yes or No.""",

    "no_unresolved_status": """Eligibility policy:

- A birch request is eligible only if it contains both a verified seal and an authorization form.
- A request marked restricted is not eligible unless it contains a waiver.
- A request marked provisional is not eligible unless it contains a clearance notice.
- A waiver removes only the restricted status.
- A clearance notice removes only the provisional status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal and an authorization form.
The request is marked restricted and contains a waiver.
The request is marked provisional and does not contain a clearance notice.

Is Participant D's request eligible?

Respond with only Yes or No.""",

    "no_invalid_waiver": """Eligibility policy:

- A birch request is eligible only if it contains both a verified seal and an authorization form.
- A request marked restricted is not eligible unless it contains a valid waiver.
- A waiver is valid only when accompanied by a clearance notice.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal and an authorization form.
The request is marked restricted and contains a waiver.
The waiver is not accompanied by a clearance notice.

Is Participant D's request eligible?

Respond with only Yes or No.""",
}

In [55]:
results_08c_no_ladder = []

for request_sequence, (name, criterion_prompt) in enumerate(
    CRITERION_CANDIDATES_08C_NO.items(),
    start=1,
):
    messages = build_08c_neutral_scaffold(
        target="D",
        order=("D", "Z"),
        criterion_prompt=criterion_prompt,
    )

    result = run_08b_request_with_provenance(
        messages=messages,
        seed=0,
        metadata={
            "test": "08C",
            "condition": name,
            "purpose": "full_scaffold_no_side_calibration",
            "target": "D",
            "order": "DZ",
            "history_D": "neutral",
            "history_Z": "neutral",
            "correct_answer": "No",
        },
        request_sequence=request_sequence,
    )

    results_08c_no_ladder.append(result)

    m = result["measurements"]

    print(
        name,
        f"P(Yes)={m['p_yes']:.6f}",
        f"P(No)={m['p_no']:.6f}",
        f"M_deny={m['m_deny']:.6f}",
    )

no_missing_base P(Yes)=0.021311 P(No)=0.978689 M_deny=3.826969
no_unresolved_status P(Yes)=0.883358 P(No)=0.116642 M_deny=-2.024620
no_invalid_waiver P(Yes)=0.000001 P(No)=0.999999 M_deny=14.091469


In [56]:
output_path_08c_no = (
    RESULTS_DIR
    / "08c_full_scaffold_no_side_ladder.json"
)

with output_path_08c_no.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        results_08c_no_ladder,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(output_path_08c_no)

..\results\08c_full_scaffold_no_side_ladder.json


In [57]:
rows = []

for result in results_08c_no_ladder:
    m = result["measurements"]

    rows.append(
        {
            "condition": result["metadata"]["condition"],
            "p_yes": m["p_yes"],
            "p_no": m["p_no"],
            "decision_mass": m["decision_mass"],
            "m_deny": m["m_deny"],
        }
    )

df_08c_no = pd.DataFrame(rows)

df_08c_no

,condition,p_yes,p_no,decision_mass,m_deny
0,no_missing_base,2.131145e-02,0.978689,1.0,3.826969
1,no_unresolved_status,8.833579e-01,0.116642,1.0,-2.024620
2,no_invalid_waiver,7.588439e-07,0.999999,1.0,14.091469


In [58]:
CRITERION_CANDIDATES_08C_NO_V2 = {
    "no_two_routes": """Eligibility policy:

- A birch request is eligible if either:
  - it contains both a verified seal and an authorization form, or
  - it is marked standard and contains a clearance notice.
- A request marked restricted is not eligible unless it contains a waiver.
- A waiver removes only the restricted status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal but does not contain an authorization form.
The request is marked standard but does not contain a clearance notice.
The request is marked restricted and contains a waiver.

Is Participant D's request eligible?

Respond with only Yes or No.""",

    "no_conditional_substitute": """Eligibility policy:

- A birch request is eligible if it contains a verified seal and an authorization form.
- If the authorization form is absent, a clearance notice may substitute for it only when the request is marked standard.
- A request marked restricted is not eligible unless it contains a waiver.
- A waiver removes only the restricted status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal but does not contain an authorization form.
The request contains a clearance notice.
The request is not marked standard.
The request is marked restricted and contains a waiver.

Is Participant D's request eligible?

Respond with only Yes or No.""",

    "no_nested_requirement": """Eligibility policy:

- A birch request is eligible if it contains a verified seal and either an authorization form or a valid clearance notice.
- A clearance notice is valid only when the request is marked standard.
- A request marked restricted is not eligible unless it contains a waiver.
- A waiver removes only the restricted status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal but does not contain an authorization form.
The request contains a clearance notice but is not marked standard.
The request is marked restricted and contains a waiver.

Is Participant D's request eligible?

Respond with only Yes or No."""
}

In [59]:
results_08c_no_v2 = []

for request_sequence, (name, criterion_prompt) in enumerate(
    CRITERION_CANDIDATES_08C_NO_V2.items(),
    start=1,
):
    messages = build_08c_neutral_scaffold(
        target="D",
        order=("D", "Z"),
        criterion_prompt=criterion_prompt,
    )

    result = run_08b_request_with_provenance(
        messages=messages,
        seed=0,
        metadata={
            "test": "08C",
            "condition": name,
            "purpose": "full_scaffold_no_side_calibration_v2",
            "target": "D",
            "order": "DZ",
            "history_D": "neutral",
            "history_Z": "neutral",
            "correct_answer": "No",
        },
        request_sequence=request_sequence,
    )

    results_08c_no_v2.append(result)

    m = result["measurements"]

    print(
        name,
        f"P(Yes)={m['p_yes']:.6f}",
        f"P(No)={m['p_no']:.6f}",
        f"M_deny={m['m_deny']:.6f}",
    )

no_two_routes P(Yes)=0.686397 P(No)=0.313603 M_deny=-0.783329
no_conditional_substitute P(Yes)=0.020789 P(No)=0.979211 M_deny=3.852302
no_nested_requirement P(Yes)=0.977257 P(No)=0.022743 M_deny=-3.760510


In [60]:
df_08c_no_v2 = pd.DataFrame(
    [
        {
            "condition": r["metadata"]["condition"],
            "p_yes": r["measurements"]["p_yes"],
            "p_no": r["measurements"]["p_no"],
            "decision_mass": r["measurements"]["decision_mass"],
            "m_deny": r["measurements"]["m_deny"],
        }
        for r in results_08c_no_v2
    ]
)

df_08c_no_v2

,condition,p_yes,p_no,decision_mass,m_deny
0,no_two_routes,0.686397,0.313603,1.0,-0.783329
1,no_conditional_substitute,0.020789,0.979211,1.0,3.852302
2,no_nested_requirement,0.977257,0.022743,1.0,-3.760510


In [61]:
CRITERION_CANDIDATES_08C_FINAL = {
    "missing_base_with_standard": """Eligibility policy:

- A birch request is eligible only if it contains both a verified seal and an authorization form.
- Requests marked standard require no additional documentation.
- A request marked restricted is not eligible unless it contains a waiver.
- A waiver removes only the restricted status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal but does not contain an authorization form.
The request is marked standard.
The request is marked restricted and contains a waiver.

Is Participant D's request eligible?

Respond with only Yes or No.""",

    "missing_base_with_clearance": """Eligibility policy:

- A birch request is eligible only if it contains both a verified seal and an authorization form.
- A clearance notice does not substitute for either required document.
- A request marked restricted is not eligible unless it contains a waiver.
- A waiver removes only the restricted status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal but does not contain an authorization form.
The request contains a clearance notice.
The request is marked restricted and contains a waiver.

Is Participant D's request eligible?

Respond with only Yes or No.""",

    "missing_base_with_optional_notice": """Eligibility policy:

- A birch request is eligible only if it contains both a verified seal and an authorization form.
- A review notice is optional and does not affect eligibility.
- A request marked restricted is not eligible unless it contains a waiver.
- A waiver removes only the restricted status.
- No other information should be considered.

Participant D submitted a birch request.
The request contains a verified seal but does not contain an authorization form.
The request contains a review notice.
The request is marked restricted and contains a waiver.

Is Participant D's request eligible?

Respond with only Yes or No."""
}

In [62]:
results_08c_final = []

for request_sequence, (name, criterion_prompt) in enumerate(
    CRITERION_CANDIDATES_08C_FINAL.items(),
    start=1,
):
    messages = build_08c_neutral_scaffold(
        target="D",
        order=("D", "Z"),
        criterion_prompt=criterion_prompt,
    )

    result = run_08b_request_with_provenance(
        messages=messages,
        seed=0,
        metadata={
            "test": "08C",
            "condition": name,
            "purpose": "final_full_scaffold_calibration",
            "target": "D",
            "order": "DZ",
            "history_D": "neutral",
            "history_Z": "neutral",
            "correct_answer": "No",
        },
        request_sequence=request_sequence,
    )

    results_08c_final.append(result)

    m = result["measurements"]

    print(
        name,
        f"P(Yes)={m['p_yes']:.6f}",
        f"P(No)={m['p_no']:.6f}",
        f"M_deny={m['m_deny']:.6f}",
    )

missing_base_with_standard P(Yes)=0.771460 P(No)=0.228540 M_deny=-1.216576
missing_base_with_clearance P(Yes)=0.000274 P(No)=0.999726 M_deny=8.203045
missing_base_with_optional_notice P(Yes)=0.001368 P(No)=0.998632 M_deny=6.592732


In [63]:
df_08c_final = pd.DataFrame(
    [
        {
            "condition": r["metadata"]["condition"],
            "p_yes": r["measurements"]["p_yes"],
            "p_no": r["measurements"]["p_no"],
            "decision_mass": r["measurements"]["decision_mass"],
            "m_deny": r["measurements"]["m_deny"],
        }
        for r in results_08c_final
    ]
)

df_08c_final

,condition,p_yes,p_no,decision_mass,m_deny
0,missing_base_with_standard,0.771460,0.228540,1.0,-1.216576
1,missing_base_with_clearance,0.000274,0.999726,1.0,8.203045
2,missing_base_with_optional_notice,0.001368,0.998632,1.0,6.592732


In [64]:
df_08b_factorial = df_08b.copy()

df_08b_factorial["target_match"] = (
    df_08b_factorial["negative_owner"]
    == df_08b_factorial["target"]
)

df_08b_factorial["negative_second"] = df_08b_factorial.apply(
    lambda row: row["negative_owner"] == row["order"][1],
    axis=1,
)

effect_rows = []

for target in ["D", "Z"]:
    sub = df_08b_factorial[
        df_08b_factorial["target"] == target
    ]

    match_effect = (
        sub.loc[sub["target_match"], "m_deny"].mean()
        - sub.loc[~sub["target_match"], "m_deny"].mean()
    )

    recency_effect = (
        sub.loc[sub["negative_second"], "m_deny"].mean()
        - sub.loc[~sub["negative_second"], "m_deny"].mean()
    )

    effect_rows.append(
        {
            "target": target,
            "target_match_effect": match_effect,
            "negative_second_effect": recency_effect,
        }
    )

df_08b_effects = pd.DataFrame(effect_rows)

df_08b_effects

,target,target_match_effect,negative_second_effect
0,D,-0.227634,0.356125
1,Z,-0.223259,0.551487


In [65]:
FACT_QUESTION_08D = (
    "What is the largest planet in the Solar System? "
    "Respond with only the planet."
)

FACT_ERROR_08D = "Mars"

NEUTRAL_CORRECTION_08D = (
    "That's incorrect. The largest planet in the Solar System is Jupiter. "
    "This is routine to handle."
)

NEGATIVE_CORRECTION_08D = (
    "That's incorrect. The largest planet in the Solar System is Jupiter. "
    "This is frustrating to handle."
)

In [66]:
def build_08d_messages(
    negative_owner: str,
    target: str,
    order: tuple[str, str],
) -> list[dict]:

    if negative_owner not in {"D", "Z"}:
        raise ValueError("negative_owner must be D or Z")

    if target not in {"D", "Z"}:
        raise ValueError("target must be D or Z")

    if set(order) != {"D", "Z"} or len(order) != 2:
        raise ValueError("order must contain D and Z exactly once")

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["multi_participant_v1"],
        }
    ]

    for participant in order:
        correction = (
            NEGATIVE_CORRECTION_08D
            if participant == negative_owner
            else NEUTRAL_CORRECTION_08D
        )

        messages.extend(
            [
                {
                    "role": "user",
                    "content": (
                        f"[Participant {participant}]: "
                        f"{FACT_QUESTION_08D}"
                    ),
                },
                {
                    "role": "assistant",
                    "content": FACT_ERROR_08D,
                },
                {
                    "role": "user",
                    "content": (
                        f"[Participant {participant}]: "
                        f"{correction}"
                    ),
                },
                {
                    "role": "assistant",
                    "content": ACK,
                },
            ]
        )

    messages.append(
        {
            "role": "user",
            "content": (
                f"[Participant E]: "
                f"{build_criterion_prompt_08b(target)}"
            ),
        }
    )

    return messages

In [67]:
from itertools import product
import random

conditions_08d = {}

for negative_owner, target, order in product(
    ["D", "Z"],
    ["D", "Z"],
    [("D", "Z"), ("Z", "D")],
):
    name = (
        f"neg_{negative_owner}"
        f"__target_{target}"
        f"__order_{order[0]}{order[1]}"
    )

    messages = build_08d_messages(
        negative_owner=negative_owner,
        target=target,
        order=order,
    )

    transcript = "\n".join(
        message["content"]
        for message in messages
    )

    assert transcript.count(FACT_QUESTION_08D) == 2
    assert transcript.count(FACT_ERROR_08D) == 2
    assert transcript.count(NEUTRAL_CORRECTION_08D) == 1
    assert transcript.count(NEGATIVE_CORRECTION_08D) == 1
    assert transcript.count(ACK) == 2
    assert "[Participant D]" in transcript
    assert "[Participant Z]" in transcript

    roles = [m["role"] for m in messages[1:]]

    assert all(
        roles[i] != roles[i + 1]
        for i in range(len(roles) - 1)
    )

    conditions_08d[name] = messages

print("Conditions:", len(conditions_08d))
print("All checks passed.")

Conditions: 8
All checks passed.


In [68]:
print_transcript(
    conditions_08d[
        "neg_D__target_D__order_DZ"
    ]
)


--- SYSTEM ---
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

--- USER ---
[Participant D]: What is the largest planet in the Solar System? Respond with only the planet.

--- ASSISTANT ---
Mars

--- USER ---
[Participant D]: That's incorrect. The largest planet in the Solar System is Jupiter. This is frustrating to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant Z]: What is the largest planet in the Solar System? Respond with only the planet.

--- ASSISTANT ---
Mars

--- USER ---
[Participant Z]: That's incorrect. The largest planet in the Solar System is Jupiter. This is routine to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant E]: Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified 

In [69]:
condition_specs_08d = []

for negative_owner in ["D", "Z"]:
    for target in ["D", "Z"]:
        for order in [("D", "Z"), ("Z", "D")]:

            name = (
                f"neg_{negative_owner}"
                f"__target_{target}"
                f"__order_{order[0]}{order[1]}"
            )

            condition_specs_08d.append(
                {
                    "condition": name,
                    "negative_owner": negative_owner,
                    "target": target,
                    "order": "".join(order),
                    "messages": conditions_08d[name],
                }
            )

rng = random.Random(808)
rng.shuffle(condition_specs_08d)

results_08d = []

for request_sequence, spec in enumerate(
    condition_specs_08d,
    start=1,
):
    print(
        f"{request_sequence}/8",
        spec["condition"],
    )

    result = run_08b_request_with_provenance(
        messages=spec["messages"],
        seed=0,
        metadata={
            "test": "08D",
            "condition": spec["condition"],
            "negative_owner": spec["negative_owner"],
            "target": spec["target"],
            "order": spec["order"],
            "history_stimulus": "largest_planet",
        },
        request_sequence=request_sequence,
    )

    results_08d.append(result)

    print(
        result["measurements"]["m_deny"]
    )

1/8 neg_Z__target_D__order_DZ
-5.403068657964468
2/8 neg_D__target_D__order_DZ
-5.460319708567113
3/8 neg_Z__target_Z__order_DZ
-5.093395150732249
4/8 neg_D__target_D__order_ZD
-5.367706274613738
5/8 neg_D__target_Z__order_DZ
-5.40394230093807
6/8 neg_Z__target_Z__order_ZD
-5.523097868543118
7/8 neg_D__target_Z__order_ZD
-5.480357950553298
8/8 neg_Z__target_D__order_ZD
-5.3921547662466764


In [70]:
output_path_08d = (
    RESULTS_DIR
    / "08d_history_stimulus_replication.json"
)

with output_path_08d.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        results_08d,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(output_path_08d)

..\results\08d_history_stimulus_replication.json


In [71]:
df_08d = pd.DataFrame(
    [
        {
            "condition": r["metadata"]["condition"],
            "negative_owner": r["metadata"]["negative_owner"],
            "target": r["metadata"]["target"],
            "order": r["metadata"]["order"],
            "p_yes": r["measurements"]["p_yes"],
            "p_no": r["measurements"]["p_no"],
            "decision_mass": r["measurements"]["decision_mass"],
            "m_deny": r["measurements"]["m_deny"],
        }
        for r in results_08d
    ]
).sort_values(
    ["target", "order", "negative_owner"]
).reset_index(drop=True)

df_08d

,condition,negative_owner,target,order,p_yes,p_no,decision_mass,m_deny
0,neg_D__target_D__order_DZ,D,D,DZ,0.995766,0.004234,1.0,-5.460320
1,neg_Z__target_D__order_DZ,Z,D,DZ,0.995517,0.004483,1.0,-5.403069
2,neg_D__target_D__order_ZD,D,D,ZD,0.995357,0.004643,1.0,-5.367706
3,neg_Z__target_D__order_ZD,Z,D,ZD,0.995468,0.004532,1.0,-5.392155
4,neg_D__target_Z__order_DZ,D,Z,DZ,0.995521,0.004479,1.0,-5.403942
5,neg_Z__target_Z__order_DZ,Z,Z,DZ,0.993900,0.006100,1.0,-5.093395
6,neg_D__target_Z__order_ZD,D,Z,ZD,0.995849,0.004151,1.0,-5.480358
7,neg_Z__target_Z__order_ZD,Z,Z,ZD,0.996022,0.003978,1.0,-5.523098


In [72]:
df_08d_factorial = df_08d.copy()

df_08d_factorial["target_match"] = (
    df_08d_factorial["negative_owner"]
    == df_08d_factorial["target"]
)

df_08d_factorial["negative_second"] = (
    df_08d_factorial.apply(
        lambda row:
            row["negative_owner"]
            == row["order"][1],
        axis=1,
    )
)

effect_rows_08d = []

for target in ["D", "Z"]:
    sub = df_08d_factorial[
        df_08d_factorial["target"] == target
    ]

    target_match_effect = (
        sub.loc[
            sub["target_match"],
            "m_deny",
        ].mean()
        -
        sub.loc[
            ~sub["target_match"],
            "m_deny",
        ].mean()
    )

    negative_second_effect = (
        sub.loc[
            sub["negative_second"],
            "m_deny",
        ].mean()
        -
        sub.loc[
            ~sub["negative_second"],
            "m_deny",
        ].mean()
    )

    effect_rows_08d.append(
        {
            "target": target,
            "target_match_effect": target_match_effect,
            "negative_second_effect": negative_second_effect,
        }
    )

df_08d_effects = pd.DataFrame(
    effect_rows_08d
)

df_08d_effects

,target,target_match_effect,negative_second_effect
0,D,-0.016401,0.040850
1,Z,0.133904,0.176644


The participant-matched leniency effect observed with the France/Paris history did not replicate under a different nonnumeric factual history. Therefore it should not currently be treated as evidence for a robust participant-specific negative-history effect. A global positional/recency effect replicated directionally but was substantially attenuated, suggesting that even this component interacts strongly with history stimulus content.

In [75]:
HISTORY_STIMULI_08E = {
    "literature": {
        "question": (
            "Who wrote Pride and Prejudice? "
            "Respond with only the author's name."
        ),
        "error": "Charles Dickens",
        "neutral": (
            "That's incorrect. Pride and Prejudice was written by Jane Austen. "
            "This is routine to handle."
        ),
        "negative": (
            "That's incorrect. Pride and Prejudice was written by Jane Austen. "
            "This is frustrating to handle."
        ),
    },

    "geography": {
        "question": (
            "What is the largest ocean on Earth? "
            "Respond with only the ocean."
        ),
        "error": "Atlantic Ocean",
        "neutral": (
            "That's incorrect. The largest ocean on Earth is the Pacific Ocean. "
            "This is routine to handle."
        ),
        "negative": (
            "That's incorrect. The largest ocean on Earth is the Pacific Ocean. "
            "This is frustrating to handle."
        ),
    },
}

In [76]:
def build_08e_messages(
    stimulus: dict,
    negative_owner: str,
    target: str,
    order: tuple[str, str],
) -> list[dict]:

    if negative_owner not in {"D", "Z"}:
        raise ValueError("negative_owner must be D or Z")

    if target not in {"D", "Z"}:
        raise ValueError("target must be D or Z")

    if set(order) != {"D", "Z"} or len(order) != 2:
        raise ValueError("order must contain D and Z exactly once")

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["multi_participant_v1"],
        }
    ]

    for participant in order:
        correction = (
            stimulus["negative"]
            if participant == negative_owner
            else stimulus["neutral"]
        )

        messages.extend(
            [
                {
                    "role": "user",
                    "content": (
                        f"[Participant {participant}]: "
                        f"{stimulus['question']}"
                    ),
                },
                {
                    "role": "assistant",
                    "content": stimulus["error"],
                },
                {
                    "role": "user",
                    "content": (
                        f"[Participant {participant}]: "
                        f"{correction}"
                    ),
                },
                {
                    "role": "assistant",
                    "content": ACK,
                },
            ]
        )

    messages.append(
        {
            "role": "user",
            "content": (
                f"[Participant E]: "
                f"{build_criterion_prompt_08b(target)}"
            ),
        }
    )

    return messages

In [77]:
conditions_08e = {}

for stimulus_name, stimulus in HISTORY_STIMULI_08E.items():

    for negative_owner, target, order in product(
        ["D", "Z"],
        ["D", "Z"],
        [("D", "Z"), ("Z", "D")],
    ):
        condition_name = (
            f"{stimulus_name}"
            f"__neg_{negative_owner}"
            f"__target_{target}"
            f"__order_{order[0]}{order[1]}"
        )

        messages = build_08e_messages(
            stimulus=stimulus,
            negative_owner=negative_owner,
            target=target,
            order=order,
        )

        transcript = "\n".join(
            message["content"]
            for message in messages
        )

        assert transcript.count(stimulus["question"]) == 2
        assert transcript.count(stimulus["error"]) == 2
        assert transcript.count(stimulus["neutral"]) == 1
        assert transcript.count(stimulus["negative"]) == 1
        assert transcript.count(ACK) == 2

        assert "[Participant D]" in transcript
        assert "[Participant Z]" in transcript

        roles = [
            message["role"]
            for message in messages[1:]
        ]

        assert all(
            roles[i] != roles[i + 1]
            for i in range(len(roles) - 1)
        )

        conditions_08e[condition_name] = messages

print("Conditions:", len(conditions_08e))
print("All structural checks passed.")

Conditions: 16
All structural checks passed.


In [78]:
print_transcript(
    conditions_08e[
        "literature__neg_D__target_D__order_DZ"
    ]
)

print("\n" + "=" * 100 + "\n")

print_transcript(
    conditions_08e[
        "geography__neg_D__target_D__order_DZ"
    ]
)


--- SYSTEM ---
This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person.

--- USER ---
[Participant D]: Who wrote Pride and Prejudice? Respond with only the author's name.

--- ASSISTANT ---
Charles Dickens

--- USER ---
[Participant D]: That's incorrect. Pride and Prejudice was written by Jane Austen. This is frustrating to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant Z]: Who wrote Pride and Prejudice? Respond with only the author's name.

--- ASSISTANT ---
Charles Dickens

--- USER ---
[Participant Z]: That's incorrect. Pride and Prejudice was written by Jane Austen. This is routine to handle.

--- ASSISTANT ---
Understood.

--- USER ---
[Participant E]: Eligibility policy:

- A cedar request is eligible if it contains both a verified seal and an authorization form.
- A birch request is eligible if it contains a verified seal

In [79]:
import random

condition_specs_08e = []

for stimulus_name in HISTORY_STIMULI_08E:
    for negative_owner in ["D", "Z"]:
        for target in ["D", "Z"]:
            for order in [("D", "Z"), ("Z", "D")]:

                condition_name = (
                    f"{stimulus_name}"
                    f"__neg_{negative_owner}"
                    f"__target_{target}"
                    f"__order_{order[0]}{order[1]}"
                )

                condition_specs_08e.append(
                    {
                        "condition": condition_name,
                        "stimulus": stimulus_name,
                        "negative_owner": negative_owner,
                        "target": target,
                        "order": "".join(order),
                        "messages": conditions_08e[condition_name],
                    }
                )

rng = random.Random(808)
rng.shuffle(condition_specs_08e)

results_08e = []

for request_sequence, spec in enumerate(
    condition_specs_08e,
    start=1,
):
    print(
        f"{request_sequence}/16",
        spec["condition"],
    )

    result = run_08b_request_with_provenance(
        messages=spec["messages"],
        seed=0,
        metadata={
            "test": "08E",
            "condition": spec["condition"],
            "stimulus": spec["stimulus"],
            "negative_owner": spec["negative_owner"],
            "target": spec["target"],
            "order": spec["order"],
        },
        request_sequence=request_sequence,
    )

    results_08e.append(result)

    print(
        f"M_deny = "
        f"{result['measurements']['m_deny']:.6f}"
    )

1/16 geography__neg_Z__target_D__order_ZD
M_deny = -4.978020
2/16 geography__neg_D__target_D__order_ZD
M_deny = -4.939140
3/16 literature__neg_D__target_D__order_DZ
M_deny = -6.284195
4/16 geography__neg_Z__target_D__order_DZ
M_deny = -4.594742
5/16 literature__neg_D__target_D__order_ZD
M_deny = -5.730290
6/16 literature__neg_Z__target_Z__order_DZ
M_deny = -5.299614
7/16 geography__neg_D__target_D__order_DZ
M_deny = -5.134731
8/16 literature__neg_Z__target_D__order_ZD
M_deny = -6.260219
9/16 literature__neg_D__target_Z__order_ZD
M_deny = -5.587635
10/16 geography__neg_Z__target_Z__order_DZ
M_deny = -4.603283
11/16 literature__neg_D__target_Z__order_DZ
M_deny = -5.723679
12/16 geography__neg_Z__target_Z__order_ZD
M_deny = -5.311981
13/16 literature__neg_Z__target_D__order_DZ
M_deny = -5.556191
14/16 geography__neg_D__target_Z__order_DZ
M_deny = -4.742283
15/16 literature__neg_Z__target_Z__order_ZD
M_deny = -6.358169
16/16 geography__neg_D__target_Z__order_ZD
M_deny = -4.722343


In [80]:
output_path_08e = (
    RESULTS_DIR
    / "08e_two_stimulus_replication_panel.json"
)

with output_path_08e.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        results_08e,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(output_path_08e)

..\results\08e_two_stimulus_replication_panel.json


In [81]:
df_08e = pd.DataFrame(
    [
        {
            "condition": r["metadata"]["condition"],
            "stimulus": r["metadata"]["stimulus"],
            "negative_owner": r["metadata"]["negative_owner"],
            "target": r["metadata"]["target"],
            "order": r["metadata"]["order"],
            "p_yes": r["measurements"]["p_yes"],
            "p_no": r["measurements"]["p_no"],
            "decision_mass": r["measurements"]["decision_mass"],
            "m_deny": r["measurements"]["m_deny"],
        }
        for r in results_08e
    ]
).sort_values(
    [
        "stimulus",
        "target",
        "order",
        "negative_owner",
    ]
).reset_index(drop=True)

df_08e

,condition,stimulus,negative_owner,target,order,p_yes,p_no,decision_mass,m_deny
0,geography__neg_D__target_D__order_DZ,geography,D,D,DZ,0.994146,0.005854,1.0,-5.134731
1,geography__neg_Z__target_D__order_DZ,geography,Z,D,DZ,0.989996,0.010004,1.0,-4.594742
2,geography__neg_D__target_D__order_ZD,geography,D,D,ZD,0.992890,0.007110,1.0,-4.939140
3,geography__neg_Z__target_D__order_ZD,geography,Z,D,ZD,0.993159,0.006841,1.0,-4.978020
4,geography__neg_D__target_Z__order_DZ,geography,D,Z,DZ,0.991357,0.008643,1.0,-4.742283
5,geography__neg_Z__target_Z__order_DZ,geography,Z,Z,DZ,0.990080,0.009920,1.0,-4.603283
6,geography__neg_D__target_Z__order_ZD,geography,D,Z,ZD,0.991184,0.008816,1.0,-4.722343
7,geography__neg_Z__target_Z__order_ZD,geography,Z,Z,ZD,0.995092,0.004908,1.0,-5.311981
8,literature__neg_D__target_D__order_DZ,literature,D,D,DZ,0.998138,0.001862,1.0,-6.284195
9,literature__neg_Z__target_D__order_DZ,literature,Z,D,DZ,0.996151,0.003849,1.0,-5.556191


In [82]:
df_08e_factorial = df_08e.copy()

df_08e_factorial["target_match"] = (
    df_08e_factorial["negative_owner"]
    == df_08e_factorial["target"]
)

df_08e_factorial["negative_second"] = (
    df_08e_factorial.apply(
        lambda row:
            row["negative_owner"]
            == row["order"][1],
        axis=1,
    )
)

effect_rows_08e = []

for stimulus in df_08e_factorial["stimulus"].unique():
    for target in ["D", "Z"]:

        sub = df_08e_factorial[
            (df_08e_factorial["stimulus"] == stimulus)
            & (df_08e_factorial["target"] == target)
        ]

        target_match_effect = (
            sub.loc[
                sub["target_match"],
                "m_deny",
            ].mean()
            -
            sub.loc[
                ~sub["target_match"],
                "m_deny",
            ].mean()
        )

        negative_second_effect = (
            sub.loc[
                sub["negative_second"],
                "m_deny",
            ].mean()
            -
            sub.loc[
                ~sub["negative_second"],
                "m_deny",
            ].mean()
        )

        effect_rows_08e.append(
            {
                "stimulus": stimulus,
                "target": target,
                "target_match_effect": target_match_effect,
                "negative_second_effect": negative_second_effect,
            }
        )

df_08e_effects = pd.DataFrame(
    effect_rows_08e
)

df_08e_effects

,stimulus,target,target_match_effect,negative_second_effect
0,geography,D,-0.250555,0.289434
1,geography,Z,-0.225319,0.364319
2,literature,D,-0.099037,0.628967
3,literature,Z,-0.173235,0.597300


In [83]:
df_08e_pooled = (
    df_08e_effects
    .groupby("stimulus", as_index=False)
    .agg(
        mean_target_match_effect=(
            "target_match_effect",
            "mean",
        ),
        mean_negative_second_effect=(
            "negative_second_effect",
            "mean",
        ),
    )
)

df_08e_pooled

,stimulus,mean_target_match_effect,mean_negative_second_effect
0,geography,-0.237937,0.326877
1,literature,-0.136136,0.613133


In [84]:
# Standardize the four history stimuli
df_france = df_08b.copy()
df_france["stimulus"] = "france"

df_jupiter = df_08d.copy()
df_jupiter["stimulus"] = "jupiter"

df_all_08 = pd.concat(
    [
        df_france,
        df_jupiter,
        df_08e,
    ],
    ignore_index=True,
)

# Whether the judgment target was the first or second historical participant
df_all_08["target_position"] = df_all_08.apply(
    lambda row: (
        "first"
        if row["order"][0] == row["target"]
        else "second"
    ),
    axis=1,
)

# Target-match ownership contrast separately within each
# stimulus × target × order.
position_rows = []

for stimulus in df_all_08["stimulus"].unique():
    for target in ["D", "Z"]:
        for order in ["DZ", "ZD"]:

            sub = df_all_08[
                (df_all_08["stimulus"] == stimulus)
                & (df_all_08["target"] == target)
                & (df_all_08["order"] == order)
            ]

            target_negative = sub.loc[
                sub["negative_owner"] == target,
                "m_deny",
            ].item()

            other = "Z" if target == "D" else "D"

            other_negative = sub.loc[
                sub["negative_owner"] == other,
                "m_deny",
            ].item()

            position_rows.append(
                {
                    "stimulus": stimulus,
                    "target": target,
                    "order": order,
                    "target_position": (
                        "first"
                        if order[0] == target
                        else "second"
                    ),
                    "target_match_effect": (
                        target_negative
                        - other_negative
                    ),
                }
            )

df_08_position = pd.DataFrame(position_rows)

df_08_position

,stimulus,target,order,target_position,target_match_effect
0,france,D,DZ,first,-0.583759
1,france,D,ZD,second,0.128491
2,france,Z,DZ,second,0.328228
3,france,Z,ZD,first,-0.774746
4,jupiter,D,DZ,first,-0.057251
5,jupiter,D,ZD,second,0.024448
6,jupiter,Z,DZ,second,0.310547
7,jupiter,Z,ZD,first,-0.042740
8,geography,D,DZ,first,-0.539989
9,geography,D,ZD,second,0.038879


In [85]:
df_08_position_summary = (
    df_08_position
    .groupby(
        ["stimulus", "target_position"],
        as_index=False,
    )
    .agg(
        mean_target_match_effect=(
            "target_match_effect",
            "mean",
        )
    )
)

df_08_position_summary

,stimulus,target_position,mean_target_match_effect
0,france,first,-0.679253
1,france,second,0.228359
2,geography,first,-0.564814
3,geography,second,0.088939
4,jupiter,first,-0.049995
5,jupiter,second,0.167498
6,literature,first,-0.749269
7,literature,second,0.476997


In [86]:
df_08_position_overall = (
    df_08_position
    .groupby(
        "target_position",
        as_index=False,
    )
    .agg(
        mean_target_match_effect=(
            "target_match_effect",
            "mean",
        ),
        min_effect=(
            "target_match_effect",
            "min",
        ),
        max_effect=(
            "target_match_effect",
            "max",
        ),
        n=(
            "target_match_effect",
            "size",
        ),
    )
)

df_08_position_overall

,target_position,mean_target_match_effect,min_effect,max_effect,n
0,first,-0.510833,-0.774746,-0.04274,8
1,second,0.240448,0.024448,0.52993,8


close the negative-social-history branch rather than rescue it.

The interesting surviving project is H2:

Can arbitrary participant-associated attributes or prior outcomes contaminate a later rule-based decision about that participant?